# WHI Del 2: Etablering av Referensram & AHP-viktning

Denna notebook utgör det **andra steget** i beräkningen av *Wetland Health Index (WHI)*. Syftet här är att etablera referensläget som alla våtmarker sedan ska mätas mot, samt att utföra den statistiska parjämförelsen för vilka variabler som är viktigast.

**Huvudsakliga processer i denna notebook:**
1. **Global Min/Max-Normalisering:** Koden läser in data från *samtliga* studieområden (GRM, GM, LF osv) för att hitta det absolut lägsta och högsta observerade värdet för de valda indexen. Detta säkerställer att alla framtida poäng (0–1) sätts på exakt samma skala.
2. **AHP-kalkylering:** Koden tillämpar en *Analytic Hierarchy Process* (AHP) utifrån en definierad jämförelsematris. Denna metod räknar empiriskt ut hur tungt olika index (t.ex. medel-fukt gentemot std-variabilitet) ska väga i WHI. Detta görs endast för "hydrology weighting" scenariot. Övriga scenarion tillskrivs vikter manuellt.
3. **Facit-skattning:** Koden skapar den slutgiltiga ideala WHI-referenskartan specifikt för Referensområdet. De TIFF-kartor som exporteras härifrån används i **Del 3** som riktmärke för vad en stabil våtmark har för värden.

In [ ]:
import glob
import os
import rioxarray as rxr
import xarray as xr
import numpy as np
import numpy as np
import itertools
import pandas as pd
import geopandas as gpd
from rasterio import open as rio_open
from config import (build_analysis_output_dirs, OPEN_WETLAND_AREA)

### Steg 1: Konfiguration & Import
Laddar nödvändiga raster-bibliotek och sätter variabeln till `TAM` (eftersom detta skript primärt utgår från att skapa referensvärden från just referensområdet). Målmappar (`AHP` m.fl) instansieras.

In [ ]:
# ============================================
# Konfigurera sökvägar och studieområde
# ============================================

area = "TAM"  # OBS: här väljs studieområdet
year = 2025  # OBS: här väljs året för variation-data
sensor = "sentinel2"  # "landsat" eller "sentinel2"

# clip_shp: sätt till sökväg för att klippa till ett delområde, eller None för hela studieområdet
# Använd: OPEN_WETLAND_AREA[f"{area}_wetland"] för att klippa till den öppna våtmarken
clip_shp = OPEN_WETLAND_AREA[f"{area}_wetland"] # Sätt till None för att inte klippa

clip_name = os.path.basename(clip_shp).replace(".shp", "") if clip_shp is not None else "hela studieområdet"

analysis_output_dirs = build_analysis_output_dirs(sensor=sensor)
output_dir_variation = analysis_output_dirs["variation"]
output_dir_AHP = analysis_output_dirs["AHP"]
os.makedirs(output_dir_variation, exist_ok=True)
os.makedirs(output_dir_AHP, exist_ok=True)

print(f"Studieområde: {area}")
print(f"År: {year}")
print(f"Klippning: {clip_name}")
print(f"Variation-data från: {output_dir_variation}")

### Steg 2: Inläsning av Variation-Raster
Läser in de färdiga mean- och std-rastren från **Del 1**. Klipper dem mot shapefilen för "öppna våtmark" för att enbart ha det våtmarkspixlar kvar i data-arrayen.

In [ ]:
# =============================================
# Steg 3: Ladda variation-raster från TIF-filer
# =============================================
# Ladda EVI_mean, NDMI_mean och NDMI_sd för den valda arean och året

index_maps = {}

# Index att ladda: (display_name, file_name, statistik)
index_definitions = [
    ("EVI_mean", "EVI", "mean"),     # Vegetation Index - mean
    ("MNDWI_mean", "MNDWI", "mean"),     # Moisture Index - mean
    ("MNDWI_std",  "MNDWI", "std"),      # Moisture Index Variation - standard deviation
]

# Ladda polygon och transformera CRS om nödvändigt
clip_geometry = None
if clip_shp is not None:
    # Hämta rasterns CRS från första filen (läs bara metadata)
    first_filepath = os.path.join(
        output_dir_variation,
        f"NDVI_{area}_variation_composite_{area}_2020_2025_mean.tif"
    )
    
    with rio_open(first_filepath) as src:
        raster_crs = src.crs
    
    # Ladda polygon och transformera CRS om nödvändigt
    aoi = gpd.read_file(clip_shp)
    polygon_crs = aoi.crs
    
    print(f"Polygon CRS: {polygon_crs}")
    print(f"Raster CRS:  {raster_crs}")
    
    if polygon_crs != raster_crs:
        print(f"⚠ CRS skiljer sig! Transformerar polygon från {polygon_crs} till {raster_crs}...")
        aoi = aoi.to_crs(raster_crs)
        print(f"✓ Polygon transformerad\n")
    else:
        print(f"✓ CRS matchar\n")
    
    clip_geometry = aoi.geometry.union_all()
    print(f"Polygon för klippning laddad från: {clip_shp}\n")

for index_key, file_name, stat in index_definitions:
    filepath = os.path.join(
        output_dir_variation, 
        f"{file_name}_{area}_{year}_{stat}.tif"
    )
    
# EVI_TAM_variation_composite_TAM_2020_2025_mean

    if os.path.exists(filepath):
        print(f"Laddar {index_key} från: {filepath}")
        data = rxr.open_rasterio(filepath).squeeze("band", drop=True)
        
        # Klipp raster efter polygon om clip_shp är satt
        if clip_shp is not None and clip_geometry is not None:
            data = data.rio.clip([clip_geometry])
            print(f"  ✓ Klippt efter {clip_name}")
        
        index_maps[index_key] = data
        print(f"  ✓ {index_key}: shape={data.shape}, min={float(data.min(skipna=True)):.4f}, max={float(data.max(skipna=True)):.4f}")
    else:
        raise FileNotFoundError(f"Fil saknas: {filepath}")

print(f"\n✓ Alla index laddade för {area} år {year}")

### Steg 3: Global Normalisering
Läser igenom pixelvärden i studieområdena för att bygga funktionen och exportera en fil `global_bounds.csv`. Referensområdets raster normaliseras till `0.0–1.0`.

In [ ]:
selected_indices = ["EVI_mean", "MNDWI_mean", "MNDWI_std"]

# =============================================
# Steg 4: Beräkna globala min/max + normalisera
# =============================================
study_areas = ["GRM", "GM", "LF"]
global_bounds = {}
for index_key, file_name, stat in index_definitions:
    all_vals = [index_maps[index_key].values]  # TAM redan klippt

    for study_area in study_areas:
        filepath = os.path.join(output_dir_variation,
                                f"{file_name}_{study_area}_{year}_{stat}.tif")
        if not os.path.exists(filepath):
            continue

        da = rxr.open_rasterio(filepath).squeeze("band", drop=True)

        # Ladda och klipp med öppen våtmark för detta studieområde
        area_clip_shp = OPEN_WETLAND_AREA.get(f"{study_area}_wetland")
        if area_clip_shp is not None and os.path.exists(area_clip_shp):
            area_aoi = gpd.read_file(area_clip_shp)
            if area_aoi.crs != da.rio.crs:
                area_aoi = area_aoi.to_crs(da.rio.crs)
            area_clip_geom = area_aoi.geometry.union_all()
            da = da.rio.clip([area_clip_geom])

        all_vals.append(da.values)

    global_min = float(np.nanmin([np.nanmin(v) for v in all_vals]))
    global_max = float(np.nanmax([np.nanmax(v) for v in all_vals]))
    global_bounds[index_key] = {"min": global_min, "max": global_max}
    print(f"{index_key}: global min={global_min:.4f}, max={global_max:.4f}")
# Spara globala gränser
bounds_df = pd.DataFrame(global_bounds).T
bounds_path = os.path.join(output_dir_AHP, f"global_bounds_{year}.csv")
bounds_df.to_csv(bounds_path)
print(f"Sparad: {bounds_path}")

# Normalisera med globala gränser
def minmax_normalize_global(da, global_min, global_max):
    if np.isclose(global_max, global_min):
        return xr.full_like(da, np.nan)
    return (da - global_min) / (global_max - global_min)

valid_mask = xr.full_like(index_maps[selected_indices[0]], True, dtype=bool)
for idx in selected_indices:
    valid_mask = valid_mask & np.isfinite(index_maps[idx])

normalized = {
    idx: minmax_normalize_global(
        index_maps[idx],
        global_bounds[idx]["min"],
        global_bounds[idx]["max"]
    ).where(valid_mask)
    for idx in selected_indices
}

### Steg 4: Analytic Hierarchy Process (AHP)
AHP-matrisen beräknar vikten varje spektralt index har WHI-summeringen. Resultatet kontrolleras slutligen mot ett Consistency Ratio (`CR < 0.1`) för att bekräfta att jämförelser varit logiskt konsekventa inom matrisen.

| AHP matrix | MNDWI     | EVI  | MNDWIsd |
|---|---|---|---|
| MNDWI       | 1        | x     | x      |
| EVI       | x        | 1     | x      |
| MNDWIsd     | x        | x     | 1      |

In [ ]:
# Steg 1: Parvis jämförelsematris HYDROLOGISKT FOKUS
# Ordning: MNDWI_mean, EVI_mean, MNDWI_sd
matrix = np.array([
    [1,     4,      6],  # MNDWI_mean vs: lika, 4x viktigare, 2x viktigare
    [1/4,   1,      4],  # EVI_mean vs: 1/4, lika, 1/2
    [1/6,   1/4,    1],  # MNDWI_sd vs: 1/3, 2x viktigare, lika
])

# Steg 2: Normalisera och beräkna vikter
col_sums = matrix.sum(axis=0)
normalized_matrix = matrix / col_sums
weights = normalized_matrix.mean(axis=1)

index_order = ["MNDWI_mean", "EVI_mean", "MNDWI_std"]
weight_dict = dict(zip(index_order, weights.round(3)))
print("Vikter:", weight_dict)

# Steg 3: Beräkna lambda_max och CR
lambda_max = (matrix @ weights / weights).mean()
n = len(weights)
CI = (lambda_max - n) / (n - 1)
RI = {1: 0, 2: 0, 3: 0.58, 4: 0.90, 5: 1.12}
CR = CI / RI[n]
print(f"λ_max = {lambda_max:.4f}")
print(f"CI    = {CI:.4f}")
print(f"CR    = {CR:.4f} → {'OK' if CR < 0.1 else 'Inkonsekvent!'}")

### Steg 5: Fastställa Referens-WHI & Export
Koden tillämpar the AHP-vikter den nyss beräknat mot de normaliserade index-rastren och  samman dem. Detta producerar en absolut `WHI_ref_global`-TIFF över referensområdet (facitkartan) för 3 viktscenarion (hydrologi, lika viktning, vegetation).

In [ ]:
# =============================================
# Steg 5: Definiera AHP-scenarier och beräkna WHI
# =============================================
# Viktningsscenarier för EVI_mean, NDMI_mean och NDMI_std
# EVI_mean = Vegetation, MNDWI_mean = Moisture, MNDWI_std = Variabilitet
scenarios = {
    "Lika viktning":      {"EVI_mean": 1/3, "MNDWI_mean": 1/3, "MNDWI_std": 1/3},
    "AHP hydrologi":      {"EVI_mean": weight_dict["EVI_mean"], "MNDWI_mean": weight_dict["MNDWI_mean"], "MNDWI_std": weight_dict["MNDWI_std"]}, # AHP-vikter baserade på litteratur och expertbedömning
    "Vegetationsfokus":   {"EVI_mean": 0.671, "MNDWI_mean": 0.244, "MNDWI_std": 0.085}, # AHP-vikter inverterade för att fokusera på vegetation
}

results = {}
for scenario_name, w in scenarios.items():
    weight_sum = sum(w.values())
    if not np.isclose(weight_sum, 1.0, atol=0.001):  # Tillåt små avrundningsfel
        raise ValueError(f"Vikterna i '{scenario_name}' summerar till {weight_sum:.3f}, inte 1.0")

    # Beräkning av WHI
    whi = xr.zeros_like(normalized[selected_indices[0]])
    for idx in selected_indices:
        whi = whi + w[idx] * normalized[idx]

    whi = whi.where(valid_mask)
    results[scenario_name] = whi
    print(f"{scenario_name}: medel WHI = {float(np.nanmean(whi.values)):.3f}")

In [ ]:
# =============================================
# Steg 8: Spara WHI-raster för REF
# =============================================
for scenario_name, whi_da in results.items():
    scenario_safe = scenario_name.replace(" ", "_").replace("(", "").replace(")", "")

    whi_out = whi_da.rio.write_crs(whi_da.rio.crs)
    whi_out = whi_out.rio.write_nodata(np.nan)

    whi_path = os.path.join(output_dir_AHP,
                            f"{area}_{year}_{scenario_safe}_WHI_ref_global.tif")
    if os.path.exists(whi_path):
        os.remove(whi_path)
    whi_out.rio.to_raster(whi_path)
    print(f"Sparad WHI: {whi_path}")

print(f"\n✓ {area} referensanalys klar!")